In [1]:
# ============================================================
# CELL 1: IMPORTS
# ============================================================
import sys
from pathlib import Path
import json
import random

sys.path.insert(0, str(Path.cwd().parent.parent))

import numpy as np
import polars as pl
import torch
from sklearn.preprocessing import LabelEncoder
from transformers import (
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    RobertaForSequenceClassification,
    RobertaTokenizerFast,
    TrainingArguments,
)
from tqdm import tqdm
import emoji

from src.utils import (
    ALL_EMOJIS,
    EMOJI_DICT,
    DATA_PATH,
    EARLY_STOPPING_CONFIG,
    MAX_LENGTH,
    MODELS_ARTIFACTS_PATH,
    SEED,
    TRAINING_CONFIG,
    WeightedTrainer,
    compute_metrics,
    create_datasets,
    get_class_weights,
    load_and_preprocess_csv,
    make_predictions,
    plot_confusion_matrix,
    preprocess_text_en,
    print_class_distribution,
    print_classification_report,
    save_model_and_encoder,
    set_seed,
    predict_emotion,
    # Emoji augmentation
    add_emojis_to_dataset,
    get_augmentation_stats,
    print_augmentation_examples,
    validate_augmentation,
    preprocess_dataframe
)

print("✅ Tous les imports réussis!")

c:\Users\antoa\Desktop\python-project\content-monitoring\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Tous les imports réussis!


In [34]:
import json
from pathlib import Path

import numpy as np
import polars as pl
import torch

from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    set_seed,
)

from datasets import Dataset

# ---------------- CONFIG ----------------
SEED = 42
MAX_LENGTH = 128
TARGET_PER_CLASS = 6000
NUM_EPOCHS = 3
BATCH_SIZE = 8

ARTIFACTS_DIR = MODELS_ARTIFACTS_PATH / "distilroberta_emotion_en_v2.3.0"

set_seed(SEED)
torch.set_num_threads(8)   # adapte à ton CPU


In [35]:
print(f"✅ Modèle v2.1.0 chargé depuis: {ARTIFACTS_DIR}")

✅ Modèle v2.1.0 chargé depuis: c:\Users\antoa\Desktop\python-project\content-monitoring\src\model_train\artifacts\distilroberta_emotion_en_v2.3.0


In [36]:
splits = {'train': 'train_df.csv', 'validation': 'val_df.csv', 'test': 'test_df.csv'}
df_Sp1786= pl.read_csv('hf://datasets/Sp1786/multiclass-sentiment-analysis-dataset/' + splits['train'])
df_Sp1786 = df_Sp1786.filter(pl.col("sentiment") == "neutral")
df_Sp1786 = df_Sp1786.rename({
    "label": "label_id",
    "sentiment": "label"
})
df_Sp1786 = preprocess_dataframe(df_Sp1786, text_col = "text", all_emojis = ALL_EMOJIS)

cols = ["text_clean", "label"]
df_Sp1786 = df_Sp1786[cols]

In [37]:
df_neutral = load_and_preprocess_csv(
    str(DATA_PATH /"version2_7classes_en"/"sentimentdataset.csv"),
    text_col="Text",
    all_emojis=ALL_EMOJIS
)

df_neutral = df_neutral.rename({
    "Sentiment": "label"
})

cols = ["text_clean", "label"]
df_neutral = df_neutral[cols]
df_neutral = df_neutral.with_columns(
    pl.col("label").str.strip_chars().alias("label")
)
# df_neutral = df_neutral.filter(pl.col("label") == "Neutral")
# df_neutral = df_neutral.with_columns(label=pl.col("label").replace("Neutral", "neutral"))
df_neutral = df_neutral.filter(
    pl.col("label").is_in(["Neutral", "Fear", "Love", "Surprise"])
)

# Normaliser les labels en minuscules
df_neutral = df_neutral.with_columns(
    pl.col("label").str.to_lowercase()
)


📥 Chargement: c:\Users\antoa\Desktop\python-project\content-monitoring\data\kaggle\version2_7classes_en\sentimentdataset.csv
   Dimensions: (732, 15)
   Colonnes: ['', 'Unnamed: 0', 'Text', 'Sentiment', 'Timestamp', 'User', 'Platform', 'Hashtags', 'Retweets', 'Likes', 'Country', 'Year', 'Month', 'Day', 'Hour']
✅ Preprocessing appliqué - colonne 'text_clean' créée


In [38]:
df_emotion = load_and_preprocess_csv(
    str(DATA_PATH /"version2_7classes_en"/"text.csv"),
    text_col="text",
    all_emojis=ALL_EMOJIS
)

# supprimer colonne vide + col text, on garde que text_clean et label
df_emotion = df_emotion.select(pl.all().exclude("", "text"))

label_map = {
    0: "sad",
    1: "joy",
    2: "love",
    3: "anger",
    4: "fear",
    5: "surprise",
}

df_emotion = df_emotion.with_columns(
    pl.col("label")
      .cast(pl.Utf8)
      .replace(label_map)
      .alias("label")
)
cols = ["text_clean", "label"]
df_emotion = df_emotion[cols]

📥 Chargement: c:\Users\antoa\Desktop\python-project\content-monitoring\data\kaggle\version2_7classes_en\text.csv
   Dimensions: (416809, 3)
   Colonnes: ['', 'text', 'label']
✅ Preprocessing appliqué - colonne 'text_clean' créée


In [39]:
df_raw = pl.concat([df_Sp1786, df_neutral, df_emotion])

In [40]:
# ============================================================
# CELL 3 — ÉQUILIBRAGE (UNDERSAMPLING)
# ============================================================
dfs = []

for label in sorted(df_raw["label"].unique()):
    df_l = df_raw.filter(pl.col("label") == label)

    if len(df_l) >= TARGET_PER_CLASS:
        df_l = df_l.sample(n=TARGET_PER_CLASS, seed=SEED)
        print(f"{label:10s}: {TARGET_PER_CLASS} (undersample)")
    else:
        print(f"{label:10s}: {len(df_l)} (keep all)")

    dfs.append(df_l)

df_balanced = pl.concat(dfs).sample(fraction=1.0, seed=SEED)

print("\n📊 Distribution équilibrée:")
print(df_balanced.group_by("label").len())

anger     : 6000 (undersample)
fear      : 6000 (undersample)
joy       : 6000 (undersample)
love      : 6000 (undersample)
neutral   : 6000 (undersample)
sad       : 6000 (undersample)
surprise  : 6000 (undersample)

📊 Distribution équilibrée:
shape: (7, 2)
┌──────────┬──────┐
│ label    ┆ len  │
│ ---      ┆ ---  │
│ str      ┆ u32  │
╞══════════╪══════╡
│ anger    ┆ 6000 │
│ joy      ┆ 6000 │
│ fear     ┆ 6000 │
│ love     ┆ 6000 │
│ surprise ┆ 6000 │
│ neutral  ┆ 6000 │
│ sad      ┆ 6000 │
└──────────┴──────┘


In [41]:
# ============================================================
# CELL 4 — LABEL ENCODING
# ============================================================
label_encoder = LabelEncoder()

df_balanced = df_balanced.with_columns(
    pl.Series(
        "label_id",
        label_encoder.fit_transform(df_balanced["label"].to_list())
    )
)

print("\n🏷️ Labels:")
for i, l in enumerate(label_encoder.classes_):
    print(f"{i} → {l}")



🏷️ Labels:
0 → anger
1 → fear
2 → joy
3 → love
4 → neutral
5 → sad
6 → surprise


In [42]:
# ============================================================
# CELL 5 — SPLIT STRATIFIÉ 80 / 20
# ============================================================
train_parts, test_parts = [], []

for label in df_balanced["label"].unique():
    df_l = df_balanced.filter(pl.col("label") == label)
    n_train = int(len(df_l) * 0.8)

    train_parts.append(df_l[:n_train])
    test_parts.append(df_l[n_train:])

df_train = pl.concat(train_parts).sample(fraction=1.0, seed=SEED)
df_test = pl.concat(test_parts).sample(fraction=1.0, seed=SEED)

print(f"\nTrain size: {len(df_train)}")
print(f"Test  size: {len(df_test)}")



Train size: 33600
Test  size: 8400


In [43]:
# ============================================================
# CELL 6 — TOKENIZATION
# ============================================================
tokenizer = RobertaTokenizerFast.from_pretrained("roberta-base")

def tokenize(batch):
    return tokenizer(
        batch["text_clean"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_ds = Dataset.from_dict({
    "text_clean": df_train["text_clean"].to_list(),
    "labels": df_train["label_id"].to_list(),
}).map(tokenize, batched=True, remove_columns=["text_clean"])

test_ds = Dataset.from_dict({
    "text_clean": df_test["text_clean"].to_list(),
    "labels": df_test["label_id"].to_list(),
}).map(tokenize, batched=True, remove_columns=["text_clean"])

Map: 100%|██████████| 8400/8400 [00:00<00:00, 38737.77 examples/s]


In [44]:
# ============================================================
# CELL 7 — MODEL
# ============================================================
model = RobertaForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=len(label_encoder.classes_)
)

print(f"\n🤖 Parameters: {model.num_parameters():,}")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



🤖 Parameters: 124,651,015


In [45]:
# ============================================================
# CELL 9 — TRAINING ARGUMENTS (CPU ONLY)
# ============================================================
training_args = TrainingArguments(
    output_dir=str(ARTIFACTS_DIR),   # ⬅️ TOUT ICI
    do_train=True,
    do_eval=True,
    learning_rate=2e-5,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    num_train_epochs=NUM_EPOCHS,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    save_steps=1000,
    eval_steps=1000,
    dataloader_num_workers=0,
    no_cuda=True,
    seed=SEED,
)


c:\Users\antoa\Desktop\python-project\content-monitoring\.venv\Lib\site-packages\transformers\training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [46]:
# ============================================================
# CELL 10 — TRAINER & TRAINING
# ============================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

trainer.train()

C:\Users\antoa\AppData\Local\Temp\ipykernel_25176\1340417818.py:4: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
100,1.947100
200,1.962400
300,1.936400
400,1.584300
500,1.100100
600,0.799500
700,0.606300
800,0.509900
900,0.517500
1000,0.463100


TrainOutput(global_step=12600, training_loss=0.26928204945155554, metrics={'train_runtime': 16839.985, 'train_samples_per_second': 5.986, 'train_steps_per_second': 0.748, 'total_flos': 2213360779822800.0, 'train_loss': 0.26928204945155554, 'epoch': 3.0})

A SAUVEGARDER AVEC PICKLES , LE MODEL, le encoder et le 

In [47]:
# ============================================================
# CELL 11 — SAUVEGARDE FINALE (TOUT AU MÊME ENDROIT)
# ============================================================

trainer.save_model(ARTIFACTS_DIR)
tokenizer.save_pretrained(ARTIFACTS_DIR)

with open(ARTIFACTS_DIR / "label_encoder.json", "w") as f:
    json.dump(label_encoder.classes_.tolist(), f, indent=2)

print("\n✅ TRAINING TERMINÉ")
print(f"📂 Tous les artefacts sont dans : {ARTIFACTS_DIR.resolve()}")




✅ TRAINING TERMINÉ
📂 Tous les artefacts sont dans : C:\Users\antoa\Desktop\python-project\content-monitoring\src\model_train\artifacts\distilroberta_emotion_en_v2.3.0


In [49]:
# ============================================================
# CELL 12 — INFÉRENCE CPU (OPTIONNEL)
# ============================================================

model.eval()
model.to("cpu")

text = "i miss all the others as well that feel that i wronged them and they will soon understand that i didnt"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH
)

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)[0]

idx = probs.argmax().item()
print(label_encoder.classes_[idx], probs[idx].item())



anger 0.9997285008430481


In [2]:
# ============================================================
# ÉTAPE 13: CHARGER LE MODÈLE v2.3.0
# ============================================================
print("\n" + "="*70)
print("📦 Chargement du modèle v2.3.0")
print("="*70)

model_path = MODELS_ARTIFACTS_PATH / "distilroberta_emotion_en_v2.3.0"

try:
    model = RobertaForSequenceClassification.from_pretrained(str(model_path))
    tokenizer = RobertaTokenizerFast.from_pretrained(str(model_path))
    print(f"✅ Modèle v2.3.0 chargé depuis: {model_path}")
except FileNotFoundError:
    print(f"❌ Modèle v2.3.0 non trouvé à: {model_path}")
    print(" Assure-toi que le fine-tuning est terminé!")
    sys.exit(1)


📦 Chargement du modèle v2.3.0
✅ Modèle v2.3.0 chargé depuis: c:\Users\antoa\Desktop\python-project\content-monitoring\src\model_train\artifacts\distilroberta_emotion_en_v2.3.0


In [3]:
label_encoder_path = model_path / "label_encoder.json"
try:
    with open(label_encoder_path, "r", encoding="utf-8") as f:
        label_encoder = json.load(f)
    print(f"🏷️ Label encoder chargé depuis: {label_encoder_path}")
except FileNotFoundError:
    print(f"❌ label_encoder.json non trouvé à: {label_encoder_path}")
    sys.exit(1)

🏷️ Label encoder chargé depuis: c:\Users\antoa\Desktop\python-project\content-monitoring\src\model_train\artifacts\distilroberta_emotion_en_v2.3.0\label_encoder.json


In [ ]:
import pickle
import json
from pathlib import Path
from transformers import RobertaForSequenceClassification, RobertaTokenizerFast

model_path = MODELS_ARTIFACTS_PATH / "distilroberta_emotion_en_v2.3.0"
label_encoder_path = model_path / "label_encoder.json"

# Charger le modèle et le tokenizer
model = RobertaForSequenceClassification.from_pretrained(str(model_path))
tokenizer = RobertaTokenizerFast.from_pretrained(str(model_path))

# Charger le label encoder
with open(label_encoder_path, "r", encoding="utf-8") as f:
    label_encoder = json.load(f)

# Sauvegarder le tout en pickle
with open("model_with_labels.pkl", "wb") as f:
    pickle.dump({
        "model_state_dict": model.state_dict(),
        "config": model.config,
        "tokenizer": tokenizer,
        "label_encoder": label_encoder
    }, f)

print("✅ Modèle + tokenizer + label_encoder sauvegardés en model_with_labels.pkl")


✅ Modèle + tokenizer + label_encoder sauvegardés en model_with_labels.pkl


In [4]:
test_sentences = [
    # JOY
    "i miss all the others as well that feel that i wronged them and they will soon understand that i didnt",
    "feel so stupid that i realise it so late",
    "i saunter through the airport terminals feeling that i have had an experience that renders the petty tribulations of everyday travel somehow far less significant",
    "i need to feel dangerous and pretty so here a striking dance pick deep in vogue minutes ago",
    "i am feeling much stronger and more confident now and by professional opinion i know that i do not have anything serious",
    "i take a shower i feel wonderful energetic and all my previous feelings about my life turn into this awesome feeling creating my life like the happiest life in the world",
    "i don t feel submissive and for the time being i ve lost interest in some bdsm stuff",
    "i would imagine this is just one of the reasons why marriage is so hard because theyll see all the good bad and ugly parts of you the parts that make it hard for you to love yourself and it feels even more awful when you feel like those parts are exposed to other people and i dont know"
    "i feel like a real fan not that i was ever a fake fan,"
    "i feel like i am actually getting something useful out of it,"
]

In [7]:
model.eval()
model.to("cpu")

inputs = tokenizer(
    test_sentences,
    return_tensors="pt",
    truncation=True,
    padding=True,
    max_length=MAX_LENGTH
)

with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1)

for i, text in enumerate(test_sentences):
    idx = probs[i].argmax().item()
    label = label_encoder[idx]
    confidence = probs[i][idx].item()

    print(f"Texte: {text}")
    print(f"→ Prédiction: {label} ({confidence:.4f})\n")


Texte: i miss all the others as well that feel that i wronged them and they will soon understand that i didnt
→ Prédiction: anger (0.9997)

Texte: feel so stupid that i realise it so late
→ Prédiction: sad (0.9997)

Texte: i saunter through the airport terminals feeling that i have had an experience that renders the petty tribulations of everyday travel somehow far less significant
→ Prédiction: anger (0.9997)

Texte: i need to feel dangerous and pretty so here a striking dance pick deep in vogue minutes ago
→ Prédiction: anger (0.9997)

Texte: i am feeling much stronger and more confident now and by professional opinion i know that i do not have anything serious
→ Prédiction: joy (0.9997)

Texte: i take a shower i feel wonderful energetic and all my previous feelings about my life turn into this awesome feeling creating my life like the happiest life in the world
→ Prédiction: joy (0.9997)

Texte: i don t feel submissive and for the time being i ve lost interest in some bdsm stuff
→ P